# Session 6: Decisions, repeats and functions

**Question of the day:** how do we make Python decide and repeat, so we never copy a formula down thousands of rows?

The rule we code is the Lakeside Mart loyalty policy: **1 point for every UGX 1,000 spent. Silver starts at 2,000 points in 12 months and Gold at 6,000 points.** Everyone else is Basic.

Work top to bottom. In every step: **predict**, **run**, **read** the output, **change one thing**. Cells marked `your code here` are yours to fill in. The check line prints OK or NOT YET.

What you keep: this notebook with a tested `loyalty_tier` function, saved with *File, Save a copy in Drive*.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Write the function

A function is a named set of steps. The parameter `spend_ugx` is the name for the value that comes in. `return` sends the answer back. The docstring says what it does.

**PREDICT first.** A customer spent 2,388,850 UGX in 12 months. How many points is that, and which tier should the function return? Work it out by hand first.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def loyalty_tier(spend_ugx):
    """Return Basic, Silver or Gold."""
    # your code here: points = the spend divided by 1000, keeping the whole part (use //)
    # your code here: if points is 6000 or more, return "Gold"; elif 2000 or more, return "Silver"; else return "Basic"
    return None

print(loyalty_tier(2388850))
check("46,660 UGX is Basic", loyalty_tier(46660) == "Basic", True)
check("2,388,850 UGX is Silver", loyalty_tier(2388850) == "Silver", True)
check("7,500,000 UGX is Gold", loyalty_tier(7500000) == "Gold", True)

**READ the output.** Does the function print the tier or return it? Run `result = loyalty_tier(2388850)` and `print(result)` to see the difference.

> My answer: 

**PREDICT first.** A Boolean expression gives True or False. With points = 2450, what does `points >= 2000 and points < 6000` give? And with points = 6000?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
points = 2450
is_silver_band = None   # your code here: points >= 2000 and points < 6000
print(is_silver_band)
points = 6000
is_silver_band_6000 = None   # your code here: the same test with points = 6000
print(is_silver_band_6000)
check("2,450 points is in the Silver band", is_silver_band is True, True)
check("exactly 6,000 points is not", is_silver_band_6000 is False, True)

**CHANGE one thing.** In `loyalty_tier`, change `>= 2000` to `> 2000`. Which input would now give a wrong tier? (Write it down, then change it back.)

> What I saw: 

---
## Step 2: Test the edges with assert

An edge case (boundary value) sits exactly where a rule changes. A test case pairs an input with the output you expect. `assert` stops with an AssertionError when a test fails.

**PREDICT first.** Which of the five test cases would fail if someone wrote `>` instead of `>=`?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
tests = [
    (0, "Basic"),
    # your code here: add the other four pairs (spend, expected tier): 1,999,999 / exactly 2,000,000 / 5,999,999 / exactly 6,000,000
]
failures = []
for spend, expected in tests:
    got = loyalty_tier(spend)
    if got != expected:
        failures.append((spend, expected, got))
print(len(failures), "failures")
for f in failures:
    print("  fails:", f)
check("five test cases", len(tests), 5)
check("no failures", len(failures) if len(tests) == 5 else None, 0)

**Optional: stop at the first failure.** An `assert` stops with an AssertionError at the first test that fails. Run the next cell once your function and tests are finished.

In [ ]:
try:
    for spend, expected in tests:
        assert loyalty_tier(spend) == expected, f"{spend} should be {expected}"
    print("All", len(tests), "tests pass")
except AssertionError as problem:
    print("AssertionError, the first failing test:", problem)

**CHANGE one thing.** Add a sixth test case of your own, for a spend of 1 UGX. What tier do you expect? Run it.

> What I saw: 

---
## Step 3: Total each customer in a loop

An accumulator is a variable that starts at 0 and collects a result as the loop goes round. `orders_2025` holds the real 2025 order values of six customers from `orders.csv`. The file is a 5 percent sample of till transactions, so these totals are far below real annual spending.

**PREDICT first.** Customer C1322 has three orders. Roughly what will the total be: thousands, hundreds of thousands, or millions of UGX?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
orders_2025 = {
    'C0009': [18400, 18060, 10200],
    'C0010': [7000, 35000, 40700],
    'C0021': [7800, 7000, 11850],
    'C1322': [2289600, 63300, 35950],
    'C0947': [2733305, 24500],
    'C0230': [7000, 16600, 48475, 2035530, 92040],
}
print(len(orders_2025), "customers")

In [ ]:
totals = {}
for customer in orders_2025:
    # your code here: start total at 0, add each order value of this customer, then store it in totals[customer]
    pass
print(totals)
grand_total = sum(totals.values()) if totals else None
check("C0009 total", totals.get("C0009"), 46660)
check("C1322 total", totals.get("C1322"), 2388850)
check("all six together", grand_total, 7502310)

**READ the output.** Why must `total = 0` sit inside the for loop (indented under the `for customer` line), and not above it? Try moving it above and read the totals.

> My answer: 

---
## Step 4: Apply the function to every customer

One loop and one function: the loop repeats the work, the function holds the rule.

**PREDICT first.** Before you run: how many of the six customers do you expect in Basic, Silver and Gold? Use the table of totals you printed.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
tiers = {}
counts = {'Basic': 0, 'Silver': 0, 'Gold': 0}
for customer in totals:
    # your code here: call loyalty_tier on the total, store it in tiers[customer], and add 1 to counts for that tier
    pass
print(tiers)
print(counts)
check("Basic customers", counts["Basic"], 3)
check("Silver customers", counts["Silver"], 3)
check("Gold customers", counts["Gold"], 0)

**CHANGE one thing.** Change the Silver boundary in `loyalty_tier` from 2000 to 2500 points and run steps 1 to 4 again. Which customers change tier? Change it back.

> What I saw: 

---
## Step 5: Read and trace a colleague's function

The function below was written by a colleague. It should return Gold for 7,500,000 UGX. Trace it by hand with that value, then write a corrected `tier_v2`.

In [ ]:
def tier_v1(spend_ugx):
    points = spend_ugx // 1000
    if points >= 2000:
        return "Silver"
    elif points >= 6000:
        return "Gold"
    else:
        return "Basic"

print(tier_v1(7500000))

**READ the output.** Which line returns first for 7,500 points, and why is the Gold test never reached?

> My answer: 

In [ ]:
def tier_v2(spend_ugx):
    # your code here: the corrected function, with a docstring
    return None

v1_failures = [s for s, e in tests if tier_v1(s) != e]
print("tier_v1 fails", len(v1_failures), "of", len(tests), "test cases")
check("tier_v1 gives Silver for 7,500,000", tier_v1(7500000) == "Silver", True)
check("tier_v2 passes every test", all(tier_v2(s) == e for s, e in tests), True)

**CHANGE one thing.** Add `(7500000, "Gold")` to `tests` and count the failures of `tier_v1` again. Does the extra test case find more faults?

> What I saw: 

**CHANGE one thing.** Stretch: the policy says points expire after 12 months. Which input would you need to add to `loyalty_tier` to enforce that? Write it down.

> What I saw: 

---
## Step 6: A spend that arrives as text

From Session 5: a string with commas is not a number. Clean it, convert it, then call the function.

**PREDICT first.** What will `loyalty_tier("2,388,850")` do? Which exception, and what does its last line say?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
spend_text = "2,388,850"
spend_clean = None   # your code here: remove the commas with replace, then convert with int()
print(spend_clean)
check("cleaned spend", spend_clean, 2388850)
check("its tier is Silver", loyalty_tier(spend_clean) == "Silver" if spend_clean else False, True)

**Optional: ask a free chat assistant** to list edge cases for `loyalty_tier`. Add each suggestion as a test case with an expected answer worked out from the policy. Your own tests decide whether its suggestion is right. Never paste personal or confidential data.

> What it suggested, and what my tests said: 

## Compare with a sheet

Put the six totals in cells A2 to A7 of a Google Sheet. In B2 write `=IF(A2>=6000000,"Gold",IF(A2>=2000000,"Silver","Basic"))` and fill down. Do the six tiers match `tiers`? If the policy moved Silver to 2,500 points, which version is quicker to change, and why?

> My comparison: 

In [ ]:
show_checks()